In [1]:
#load env which is dicom2omop_env into jupyter kernel selection and terminal
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"


In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"


def load_dicom_concepts(staging_csv_path, db_path=DB_PATH):
    """Load one DICOM2OMOP staging CSV into the OMOP concept table."""

    staging_csv_path = Path(staging_csv_path)

    if not staging_csv_path.exists():
        print(f"Error: file not found: {staging_csv_path}")
        return

    df = pd.read_csv(staging_csv_path)
    print(f"Loaded {len(df)} rows from staging CSV")
    print(f"Columns: {list(df.columns)}")
    print("\nSample rows:")
    print(df.head(3).to_string())

    print("\n--- Staging File Summary ---")
    if "vocabulary_id" in df.columns:
        print(f"Vocabularies: {df['vocabulary_id'].dropna().unique()}")
    if "domain_id" in df.columns:
        print(f"Domains: {df['domain_id'].dropna().unique()}")
    if "concept_class_id" in df.columns:
        print(f"Concept classes: {df['concept_class_id'].value_counts().to_dict()}")

    with sqlite3.connect(db_path) as conn:
        print("\n--- SQLite concept table schema ---")
        schema = pd.read_sql("PRAGMA table_info(concept);", conn)
        print(schema.to_string(index=False))

        # --- SIMPLE TYPE CLEANUP ---
        if "concept_id" in df.columns:
            df["concept_id"] = pd.to_numeric(df["concept_id"], errors="coerce")

        text_cols = [
            "concept_name",
            "domain_id",
            "vocabulary_id",
            "concept_class_id",
            "standard_concept",
            "concept_code",
            "invalid_reason",
        ]
        for col in text_cols:
            if col in df.columns:
                df[col] = df[col].astype("string")

        for col in ["valid_start_date", "valid_end_date"]:
            if col in df.columns:
                df[col] = pd.to_datetime(
                    df[col].astype(str),
                    format="%Y%m%d",
                    errors="coerce"
                ).dt.strftime("%Y-%m-%d")

        print("\n--- DataFrame dtypes after cleanup ---")
        print(df.dtypes)

        # Drop bad concept_id rows
        bad_ids = df["concept_id"].isna()
        if bad_ids.any():
            print("\n--- Rows with invalid concept_id ---")
            print(df.loc[bad_ids, ["concept_id", "concept_name"]].head(20).to_string(index=False))
            print(f"\nDropping {bad_ids.sum()} rows with invalid concept_id")
            df = df.loc[~bad_ids].copy()

        # Drop rows missing required OMOP fields
        required_cols = [
            "concept_id",
            "concept_name",
            "domain_id",
            "vocabulary_id",
            "concept_class_id",
            "concept_code",
            "valid_start_date",
            "valid_end_date",
        ]

        for col in required_cols:
            if col in df.columns:
                df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

        missing_required = df[required_cols].isna().any(axis=1)
        if missing_required.any():
            print("\n--- Rows missing required fields ---")
            print(df.loc[missing_required, required_cols].head(20).to_string(index=False))
            print(f"\nDropping {missing_required.sum()} rows missing required concept fields")
            df = df.loc[~missing_required].copy()

        if df.empty:
            print("\nNo valid rows remain after cleanup. Skipping insert.")
            return

        df["concept_id"] = df["concept_id"].astype(int)

        print(f"\nRows remaining for insert: {len(df)}")

        # Insert cleaned rows
        df.to_sql("concept", conn, if_exists="append", index=False)

        count = pd.read_sql("SELECT COUNT(*) AS n FROM concept", conn).iloc[0]["n"]
        print(f"✓ {count} total concepts now in OMOP concept table")


if __name__ == "__main__":
    staging_dir = Path(
        r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\files\OMOP CDM Staging"
    )

    if not staging_dir.exists():
        print(f"Directory not found: {staging_dir}")
        raise SystemExit(1)

    csv_file = staging_dir / "omop_table_staging_v5.csv"

    if not csv_file.exists():
        print(f"File not found: {csv_file}")
        raise SystemExit(1)

    with sqlite3.connect(DB_PATH) as conn:
        deleted = conn.execute("DELETE FROM concept").rowcount
        print(f"Cleared concept table: deleted {deleted} existing rows\n")

    print(f"Processing: {csv_file}")
    load_dicom_concepts(str(csv_file))
    print(f"Finished: {csv_file.name}")

    with sqlite3.connect(DB_PATH) as conn:
        print(pd.read_sql("SELECT COUNT(*) FROM concept", conn))
        print(pd.read_sql("SELECT * FROM concept LIMIT 10", conn))

In [ ]:
# (dicom2omop_env) PS C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project> python load_dicom_concepts.py 
# Cleared concept table: deleted 8811 existing rows

# Processing: C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\files\OMOP CDM Staging\omop_table_staging_v5.csv
# Loaded 8811 rows from staging CSV
# Columns: ['concept_id', 'concept_name', 'domain_id', 'vocabulary_id', 'concept_class_id', 'standard_concept', 'concept_code', 'valid_start_date', 'valid_end_date', 'invalid_reason']

# Sample rows:
#    concept_id            concept_name    domain_id vocabulary_id  concept_class_id  standard_concept concept_code  valid_start_date  valid_end_date  invalid_reason
# 0  2128000010           Length to End  Measurement         DICOM  DICOM Attributes               NaN     00080001          19930101        20991231             NaN
# 1  2128000011  Specific Character Set  Measurement         DICOM  DICOM Attributes               NaN     00080005          19930101        20991231             NaN
# 2  2128000012  Language Code Sequence  Measurement         DICOM  DICOM Attributes               NaN     00080006          19930101        20991231             NaN

# --- Staging File Summary ---
# Vocabularies: ['DICOM']
# Domains: ['Measurement']
# Concept classes: {'DICOM Attributes': 5183, 'DICOM Value Sets': 3628}

# --- SQLite concept table schema ---
#  cid             name    type  notnull dflt_value  pk
#    0       concept_id INTEGER        0       None   1
#    1     concept_name    TEXT        1       None   0
#    2        domain_id    TEXT        1       None   0
#    3    vocabulary_id    TEXT        1       None   0
#    4 concept_class_id    TEXT        1       None   0
#    5 standard_concept    TEXT        0       None   0
#    6     concept_code    TEXT        1       None   0
#    7 valid_start_date    TEXT        1       None   0
#    8   valid_end_date    TEXT        1       None   0
#    9   invalid_reason    TEXT        0       None   0

# --- DataFrame dtypes after cleanup ---
# concept_id                   int64
# concept_name        string[python]
# domain_id           string[python]
# vocabulary_id       string[python]
# concept_class_id    string[python]
# standard_concept    string[python]
# concept_code        string[python]
# valid_start_date            object
# valid_end_date              object
# invalid_reason      string[python]
# dtype: object

# Rows remaining for insert: 8811
# ✓ 8811 total concepts now in OMOP concept table
# Finished: omop_table_staging_v5.csv
#    COUNT(*)
# 0      8811
#    concept_id                concept_name    domain_id vocabulary_id  ... concept_code valid_start_date valid_end_date invalid_reason
# 0  2128000010               Length to End  Measurement         DICOM  ...     00080001       1993-01-01     2099-12-31           
# None
# 1  2128000011      Specific Character Set  Measurement         DICOM  ...     00080005       1993-01-01     2099-12-31           
# None
# 2  2128000012      Language Code Sequence  Measurement         DICOM  ...     00080006       1993-01-01     2099-12-31           
# None
# 3  2128000013                  Image Type  Measurement         DICOM  ...     00080008       1993-01-01     2099-12-31           
# None
# 4  2128000014            Recognition Code  Measurement         DICOM  ...     00080010       1993-01-01     2099-12-31           
# None
# 5  2128000015      Instance Creation Date  Measurement         DICOM  ...     00080012       1993-01-01     2099-12-31           
# None
# 6  2128000016      Instance Creation Time  Measurement         DICOM  ...     00080013       1993-01-01     2099-12-31           
# None
# 7  2128000017        Instance Creator UID  Measurement         DICOM  ...     00080014       1993-01-01     2099-12-31           
# None
# 8  2128000018  Instance Coercion DateTime  Measurement         DICOM  ...     00080015       1993-01-01     2099-12-31           
# None
# 9  2128000019               SOP Class UID  Measurement         DICOM  ...     00080016       1993-01-01     2099-12-31           
# None

# [10 rows x 10 columns]
# (dicom2omop_env) PS C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project> 

In [ ]:
#check that omop database is populated correctly, run in terminal with 
# get into folder with py files : cd "C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project""
# python check_database.py
import sqlite3
import pandas as pd

DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"

DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"


with sqlite3.connect(DB_PATH) as conn:

    print("\n--- Total concepts ---")
    print(pd.read_sql("SELECT COUNT(*) FROM concept", conn))

    print("\n--- Sample concepts ---")
    print(pd.read_sql("SELECT * FROM concept LIMIT 10", conn))

    print("\n--- CT / modality check ---")
    print(pd.read_sql("""
        SELECT concept_name, concept_code
        FROM concept
        WHERE LOWER(concept_name) LIKE '%computed tomography%'
           OR concept_code = 'CT'
        LIMIT 10
    """, conn))

    print("\n--- Dental-related terms ---")
    print(pd.read_sql("""
        SELECT concept_name
        FROM concept
        WHERE LOWER(concept_name) LIKE '%mandib%'
           OR LOWER(concept_name) LIKE '%maxill%'
           OR LOWER(concept_name) LIKE '%tooth%'
        LIMIT 10
    """, conn))


# (base) (dicom2omop_env) PS C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project> python check_database.py

# --- Total concepts ---
#    COUNT(*)
# 0      8811

# --- Sample concepts ---
#    concept_id                concept_name    domain_id vocabulary_id  concept_class_id standard_concept concept_code valid_start_date valid_end_date invalid_reason
# 0  2128000010               Length to End  Measurement         DICOM  DICOM Attributes             None     00080001       1993-01-01     2099-12-31           None
# 1  2128000011      Specific Character Set  Measurement         DICOM  DICOM Attributes             None     00080005       1993-01-01     2099-12-31           None
# 2  2128000012      Language Code Sequence  Measurement         DICOM  DICOM Attributes             None     00080006       1993-01-01     2099-12-31           None
# 3  2128000013                  Image Type  Measurement         DICOM  DICOM Attributes             None     00080008       1993-01-01     2099-12-31           None
# 4  2128000014            Recognition Code  Measurement         DICOM  DICOM Attributes             None     00080010       1993-01-01     2099-12-31           None
# 5  2128000015      Instance Creation Date  Measurement         DICOM  DICOM Attributes             None     00080012       1993-01-01     2099-12-31           None
# 6  2128000016      Instance Creation Time  Measurement         DICOM  DICOM Attributes             None     00080013       1993-01-01     2099-12-31           None
# 7  2128000017        Instance Creator UID  Measurement         DICOM  DICOM Attributes             None     00080014       1993-01-01     2099-12-31           None
# 8  2128000018  Instance Coercion DateTime  Measurement         DICOM  DICOM Attributes             None     00080015       1993-01-01     2099-12-31           None
# 9  2128000019               SOP Class UID  Measurement         DICOM  DICOM Attributes             None     00080016       1993-01-01     2099-12-31           None

# --- CT / modality check ---
#           concept_name concept_code
# 0  Computed tomography           CT

# --- Dental-related terms ---
#               concept_name
# 0                 Mandible
# 1                  Maxilla
# 2      Submandibular gland
# 3  Temporomandibular joint

In [3]:
# # load_dicom_concepts.py
# import pandas as pd
# import sqlite3
# from pathlib import Path

# DB_PATH = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\julie_project\omop_dental_cbct_v1.db"

# def load_dicom_concepts(staging_csv_path, db_path=DB_PATH):
#     """
#     Load the DICOM2OMOP staging CSV into the OMOP concept table.
    
#     This replaces the PostgreSQL-based load_dicom_to_omop.ipynb notebook.
#     The staging CSV contains DICOM attributes, context groups, and 
#     coded values mapped to OMOP concept format.
#     """
#     # Read the staging CSV
#     df = pd.read_csv(staging_csv_path)
#     print(f"Loaded {len(df)} rows from staging CSV")
#     print(f"Columns: {list(df.columns)}")
#     print(f"\nSample rows:")
#     print(df.head(3).to_string())
    
#     # Inspect what's in the staging file
#     print(f"\n--- Staging File Summary ---")
#     if 'vocabulary_id' in df.columns:
#         print(f"Vocabularies: {df['vocabulary_id'].unique()}")
#     if 'domain_id' in df.columns:
#         print(f"Domains: {df['domain_id'].unique()}")
#     if 'concept_class_id' in df.columns:
#         print(f"Concept classes: {df['concept_class_id'].value_counts().to_dict()}")
    
#     # Map staging CSV columns to OMOP concept table columns
#     # NOTE: You may need to adjust column names based on actual staging CSV format
#     # The staging file may have columns like:
#     #   concept_id, concept_name, domain_id, vocabulary_id, concept_class_id,
#     #   standard_concept, concept_code, valid_start_date, valid_end_date
    
#     conn = sqlite3.connect(db_path)
    
#     # Load into concept table
#     df.to_sql('concept', conn, if_exists='append', index=False)
    
#     # Verify
#     count = pd.read_sql("SELECT COUNT(*) as n FROM concept", conn).iloc[0]['n']
#     print(f"\n✓ {count} concepts now in OMOP concept table")
    
#     # Show dental-relevant concepts
#     dental_query = """
#     SELECT concept_id, concept_name, vocabulary_id, concept_class_id
#     FROM concept
#     WHERE LOWER(concept_name) LIKE '%tooth%' 
#        OR LOWER(concept_name) LIKE '%dental%'
#        OR LOWER(concept_name) LIKE '%jaw%'
#        OR LOWER(concept_name) LIKE '%mandib%'
#        OR LOWER(concept_name) LIKE '%maxill%'
#     LIMIT 20;
#     """
#     dental_concepts = pd.read_sql(dental_query, conn)
#     if len(dental_concepts) > 0:
#         print(f"\n--- Dental-relevant concepts found: ---")
#         print(dental_concepts.to_string())
#     else:
#         print(f"\n⚠ No dental-specific concepts in staging file (expected — this is DICOM vocabulary)")
    
#     # Show modality concepts (CT should be here)
#     modality_query = """
#     SELECT concept_id, concept_name, concept_code
#     FROM concept
#     WHERE LOWER(concept_name) LIKE '%computed tomography%'
#        OR concept_code = 'CT'
#        OR LOWER(concept_name) LIKE '%cone beam%'
#     LIMIT 10;
#     """
#     modality_concepts = pd.read_sql(modality_query, conn)
#     print(f"\n--- CT/CBCT modality concepts: ---")
#     print(modality_concepts.to_string() if len(modality_concepts) > 0 else "  (none found in staging — check Athena)")
    
#     conn.close()

# if __name__ == "__main__":
#     import sys
#     # Usage: python load_dicom_concepts.py "DICOM2OMOP/files/OMOP CDM Staging/omop_table_staging_v3.csv"
#     if len(sys.argv) > 1:
#         load_dicom_concepts(sys.argv[1])
#     else:
#         # Auto-find the latest staging file
#         staging_dir = Path("DICOM2OMOP/files/OMOP CDM Staging")
#         if staging_dir.exists():
#             csvs = sorted(staging_dir.glob("omop_table_staging*.csv"))
#             if csvs:
#                 print(f"Found staging file: {csvs[-1]}")
#                 load_dicom_concepts(str(csvs[-1]))
#             else:
#                 print("No staging CSVs found. Check the files/ directory.")
#         else:
#             print(f"Directory not found: {staging_dir}")
#             print("Did you clone the repo? Run: git clone https://github.com/paulnagy/DICOM2OMOP.git")